# Gold: cant_post_x_user_hist (respaldo manual de Airflow)

Hace lo mismo que la tarea de Gold del DAG: lee `post_hist` y `users_hist` de Silver, agrega los posts por usuario, los cruza con los datos del usuario, aplica reglas de negocio, agrega `fecha_cargue` y hace `MERGE INTO` en `nessie.gold.cant_post_x_user_hist` (bucket `gold`).

Reutiliza las funciones de `dags/scripts/gold_agg.py`.

**Antes de correr:** `users_hist` y `post_hist` creadas en Silver.

- `MODO = "local"`: Spark corre dentro de Jupyter (apaga el clúster para ahorrar RAM).
- `MODO = "cluster"`: usa el clúster Spark (`docker compose up -d spark-master spark-worker`); es la alternativa si el modo local se cae por memoria.

In [ ]:
import os
import socket
import sys
from pathlib import Path

# Los scripts viven en dags/scripts (la misma carpeta que usa Airflow)
for candidato in [Path("../dags/scripts"), Path("/workspace/dags/scripts"), Path("dags/scripts")]:
    if candidato.exists():
        sys.path.insert(0, str(candidato.resolve()))
        break

from pyspark.sql import functions as F
import gold_agg as gold

## Parametros

`MAX_ROWS` limita los registros de la tabla Gold (opcional); `None` = sin limite.

In [ ]:
MAX_ROWS = 200000   # None = sin limite (requiere mas RAM)
MODO = "local"    # "local" o "cluster"

## Sesion de Spark

El catalogo `nessie` y la conexion a MinIO salen del `spark-defaults.conf` de la imagen. Si cambias `MODO`, reinicia el kernel antes de crear la sesion.

In [ ]:
# Mismas opciones de JVM que el spark-defaults.conf (al pasar --driver-java-options se reemplazan)
JAVA_OPTS = "-Duser.timezone=UTC -Daws.region=us-east-1"

if MODO == "local":
    # 1 hilo y 2 GB; compilador C1 y GC Serial = JVM mas conservadora
    args = ("--master local[1] --driver-memory 2g "
            f'--driver-java-options "{JAVA_OPTS} -XX:TieredStopAtLevel=1 -XX:+UseSerialGC" pyspark-shell')
else:
    ip = socket.gethostbyname(socket.gethostname())  # para que los executors encuentren a Jupyter
    args = ("--master spark://spark-master:7077 --driver-memory 1g "
            f"--conf spark.driver.host={ip} --conf spark.executor.memory=512m --conf spark.cores.max=1 "
            f'--driver-java-options "{JAVA_OPTS}" pyspark-shell')
os.environ["PYSPARK_SUBMIT_ARGS"] = args

spark = gold.get_spark(f"gold_{gold.GOLD_TABLE}")
spark

## Lectura de Silver y agregacion

In [ ]:
users = spark.table(f"{gold.CATALOG}.{gold.SILVER_NAMESPACE}.users_hist")
post = spark.table(f"{gold.CATALOG}.{gold.SILVER_NAMESPACE}.post_hist")
print("users_hist:", users.count(), "| post_hist:", post.count())

In [ ]:
df = gold.build_gold(users, post).withColumn("fecha_cargue", F.current_timestamp())
if MAX_ROWS:
    df = df.limit(int(MAX_ROWS))
df.printSchema()
df.show(5, truncate=False)

## MERGE en Iceberg (Nessie + bucket gold)

In [ ]:
full_name = gold.merge_into_iceberg(spark, df, gold.GOLD_TABLE, key="user_id")
print("Merge listo en", full_name)

## Verificacion

In [ ]:
print("Filas en", full_name, "->", spark.table(full_name).count())
spark.sql(f"SELECT * FROM {full_name} ORDER BY total_posts DESC LIMIT 10").show(truncate=False)